In [ ]:

import os
import pandas as pd
from datasets import Dataset, Features, Value
import librosa
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

In [ ]:
#CSV_PATH = "/kaggle/input/shobdotori/Train_annotation/Bhola.csv"  # CHANGE THIS
#BASE_AUDIO_DIR = "/kaggle/input/shobdotori/Train/Bhola"  # CHANGE THIS


In [ ]:
# ==================== STEP 1: CONFIGURE ALL DIALECTS (Colab Paths) ====================
DIALECT_CONFIGS = [
    {"csv": "/content/Train_annotation/Barisal.csv", "audio": "/content/Train/Barisal"},
    {"csv": "/content/Train_annotation/Bhola.csv", "audio": "/content/Train/Bhola"},
    {"csv": "/content/Train_annotation/Bogura.csv", "audio": "/content/Train/Bogura"},
    {"csv": "/content/Train_annotation/Brahmanbaria.csv", "audio": "/content/Train/Brahmanbaria"},
    {"csv": "/content/Train_annotation/Chittagong.csv", "audio": "/content/Train/Chittagong"},
    {"csv": "/content/Train_annotation/Comilla.csv", "audio": "/content/Train/Comilla"},
    {"csv": "/content/Train_annotation/Dhaka.csv", "audio": "/content/Train/Dhaka"},
    {"csv": "/content/Train_annotation/Feni.csv", "audio": "/content/Train/Feni"},
    {"csv": "/content/Train_annotation/Jessore.csv", "audio": "/content/Train/Jessore"},
    {"csv": "/content/Train_annotation/Jhenaidah.csv", "audio": "/content/Train/Jhenaidah"},
    {"csv": "/content/Train_annotation/Khulna.csv", "audio": "/content/Train/Khulna"},
    {"csv": "/content/Train_annotation/Kushtia.csv", "audio": "/content/Train/Kushtia"},
    {"csv": "/content/Train_annotation/Lakshmipur.csv", "audio": "/content/Train/Lakshmipur"},
    {"csv": "/content/Train_annotation/Mymensingh.csv", "audio": "/content/Train/Mymensingh"},
    {"csv": "/content/Train_annotation/Natore.csv", "audio": "/content/Train/Natore"},
    {"csv": "/content/Train_annotation/Noakhali.csv", "audio": "/content/Train/Noakhali"},
    {"csv": "/content/Train_annotation/Pabna.csv", "audio": "/content/Train/Pabna"},
    {"csv": "/content/Train_annotation/Rajshahi.csv", "audio": "/content/Train/Rajshahi"},
    {"csv": "/content/Train_annotation/Rangpur.csv", "audio": "/content/Train/Rangpur"},
    {"csv": "/content/Train_annotation/Sylhet.csv", "audio": "/content/Train/Sylhet"},
]

In [ ]:
all_dfs = []

for config in DIALECT_CONFIGS:
    try:
        # Load CSV with your existing logic
        df = pd.read_csv(config["csv"], names=["audio", "text"])
        df = df.drop(0)  # Remove header row

        # Fix paths using your existing logic
        base_audio = Path(config["audio"])
        df['audio'] = df['audio'].apply(lambda x: str(base_audio / x.strip()))


        all_dfs.append(df)
        print(f"✅ Loaded {len(df)} samples from {base_audio.name}")

    except FileNotFoundError as e:
        print(f"⚠️  Skipping {config['csv']}: Not found")
    except Exception as e:
        print(f"❌ Error loading {config['csv']}: {e}")


⚠️  Skipping /content/Train_annotation/Barisal.csv: Not found
⚠️  Skipping /content/Train_annotation/Bhola.csv: Not found
⚠️  Skipping /content/Train_annotation/Bogura.csv: Not found
⚠️  Skipping /content/Train_annotation/Brahmanbaria.csv: Not found
⚠️  Skipping /content/Train_annotation/Chittagong.csv: Not found
⚠️  Skipping /content/Train_annotation/Comilla.csv: Not found
⚠️  Skipping /content/Train_annotation/Dhaka.csv: Not found
⚠️  Skipping /content/Train_annotation/Feni.csv: Not found
⚠️  Skipping /content/Train_annotation/Jessore.csv: Not found
⚠️  Skipping /content/Train_annotation/Jhenaidah.csv: Not found
⚠️  Skipping /content/Train_annotation/Khulna.csv: Not found
⚠️  Skipping /content/Train_annotation/Kushtia.csv: Not found
⚠️  Skipping /content/Train_annotation/Lakshmipur.csv: Not found
⚠️  Skipping /content/Train_annotation/Mymensingh.csv: Not found
⚠️  Skipping /content/Train_annotation/Natore.csv: Not found
⚠️  Skipping /content/Train_annotation/Noakhali.csv: Not found
⚠

In [ ]:
!unrar x train.rar
!unrar x train_annotation.rar
!unrar x test.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal

Cannot open train.rar
No such file or directory
No files to extract

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal

Cannot open train_annotation.rar
No such file or directory
No files to extract

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal

Cannot open test.rar
No such file or directory
No files to extract


After running the extraction, please execute the data loading cells again (starting with the cell defining `DIALECT_CONFIGS`).

In [ ]:
# Combine all dataframes
df = pd.concat(all_dfs, ignore_index=True)
print(f"\n🎯 Total dataset: {len(df)} samples across {len(all_dfs)} dialects")


from datasets import Features, Value

features = Features({
    "audio": Value("string"),  # FORCE string type
    "text": Value("string")
})

print(f"\n✅ Features configured: {features}")
print(f"\nSample data:\n{df.head()}")

ValueError: No objects to concatenate

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

In [ ]:
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name)

model = WhisperForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float32
).to("cuda")

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []


print(f"Model loaded. Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
from pathlib import Path
import os
import librosa


def resolve_audio_path(audio_field):
    """
    Resolve audio path - simplified for pre-resolved absolute paths
    """
    # Handle dict format (rare case)
    if isinstance(audio_field, dict) and audio_field.get("path"):
        return Path(audio_field["path"])

    # Convert to string and strip whitespace
    name = str(audio_field).strip()
    if name.lower() == "audio":
        raise FileNotFoundError("Found header value 'audio' in data.")

    # Already absolute path (your data loading ensures this)
    path = Path(name)
    if path.is_absolute() and path.is_file():
        return path

    # Fallback: try as relative to current directory
    if path.exists():
        return path

    # Last resort: search in common audio directories
    for base in [Path.cwd(), Path.cwd() / "audio"]:
        candidate = base / name
        if candidate.is_file():
            return candidate

    tried = f"{name} (absolute) | {path.cwd() / name} (relative)"
    raise FileNotFoundError(f"Audio file not found. Tried: {tried}")

In [ ]:
def prepare_dataset(batch, processor):
    audio_field = batch["audio"]
    path = resolve_audio_path(audio_field)
    if path.is_dir():
        raise FileNotFoundError(f"Got a directory, expected file: {path}")

    # Load audio
    y, sr = librosa.load(str(path), sr=16000, mono=True)

    # Process audio
    batch["input_features"] = processor.feature_extractor(
        y, sampling_rate=sr
    ).input_features[0]

    # Process text: Tokenize to create labels
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

In [ ]:
VALID_EXTS = (".wav", ".flac", ".mp3", ".ogg")
def has_valid_audio(ex):
    a = str(ex["audio"]).strip()
    return a.lower() != "audio" and a.lower().endswith(VALID_EXTS)

In [ ]:
from datasets import DatasetDict
dataset = Dataset.from_pandas(df, features=features)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({k: v.filter(has_valid_audio) for k, v in dataset.items()})
print(f"\nTrain: {len(dataset['train'])} samples")
print(f"Test: {len(dataset['test'])} samples")

In [ ]:
print("\n=== Preprocessing (handles both string and dict) ===")
dataset = dataset.map(
    prepare_dataset,
    fn_kwargs={"processor": processor},
    remove_columns=dataset["train"].column_names,
    num_proc=1
)

print(f"Done! {len(dataset['train'])} train, {len(dataset['test'])} test")


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-dialects",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    gradient_checkpointing=False,
    fp16=False,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    load_best_model_at_end=True,
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=256,
    remove_unused_columns=False,
    label_names=["labels"],
    report_to=[],
    logging_steps=10,
)


In [ ]:
torch.cuda.empty_cache()
print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    #compute_metrics=compute_metrics,
    tokenizer=processor.tokenizer,
)

print("\n=== Starting training! ===")
trainer.train()

In [ ]:
trainer.save_model("./whisper-small-dialects")

<h1>TEST</h1>

In [ ]:
!pip install -q python-Levenshtein

In [ ]:
import Levenshtein
import pandas as pd
from pathlib import Path

def calculate_similarity(reference, prediction):
    """
    Calculate similarity score using your exact formula:
    Similarity = 1.0 - (Levenshtein Distance / Max(Reference Length, Prediction Length))
    """
    # Handle empty cases
    if not reference or not prediction:
        return 0.0

    # Calculate Levenshtein distance
    distance = Levenshtein.distance(reference, prediction)

    # Get maximum length
    max_length = max(len(reference), len(prediction))

    # Avoid division by zero
    if max_length == 0:
        return 0.0

    # Calculate similarity
    similarity = 1.0 - (distance / max_length)

    return max(0.0, similarity)  # Ensure non-negative

In [ ]:
def evaluate_test_set(model, processor, test_csv_path, test_audio_dir):
    """
    Evaluate model on test set and calculate final score

    Args:
        model: Your trained Whisper model
        processor: Whisper processor
        test_csv_path: Path to CSV with columns [audio, text]
        test_audio_dir: Directory containing test audio files

    Returns:
        results_df: DataFrame with [audio, reference, prediction, similarity]
        final_score: Average similarity score
    """
    # Load test data
    test_df = pd.read_csv(test_csv_path, names=["audio", "text"])
    test_df = test_df.drop(0)  # Remove header row

    # Fix audio paths
    test_df['audio'] = test_df['audio'].apply(lambda x: str(Path(test_audio_dir) / x.strip()))

    results = []
    total_similarity = 0.0

    print("="*80)
    print("🔍 EVALUATING ON TEST SET")
    print("="*80)
    print(f"Total test samples: {len(test_df)}")

    # Process each test sample
    for idx, row in test_df.iterrows():
        audio_path = row['audio']
        reference = str(row['text']).strip()

        # Verify audio file exists
        if not Path(audio_path).exists():
            print(f"❌ Audio not found: {audio_path}")
            prediction = "[AUDIO NOT FOUND]"
            similarity = 0.0
        else:
            # Transcribe
            prediction = transcribe_audio(audio_path)

            # Calculate similarity
            similarity = calculate_similarity(reference, prediction)
            total_similarity += similarity

        results.append({
            'audio_path': Path(audio_path).name,
            'reference': reference,
            'prediction': prediction,
            'similarity': similarity
        })

        # Progress bar
        if (idx + 1) % 10 == 0:
            print(f"  → Processed {idx + 1}/{len(test_df)}...")

    # Calculate final score
    final_score = total_similarity / len(test_df)

    # Create DataFrame
    results_df = pd.DataFrame(results)

    # Display results
    print(f"\n{'='*80}")
    print(f"✅ EVALUATION COMPLETE")
    print(f"📊 Final Score: {final_score:.4f}")
    print(f"📈 Average Similarity: {final_score*100:.2f}%")
    print(f"{'='*80}\n")

    # Show worst predictions
    print("📉 WORST PREDICTIONS (Lowest Similarity):")
    worst = results_df.nsmallest(5, 'similarity')
    for idx, row in worst.iterrows():
        print(f"\nSimilarity: {row['similarity']:.3f}")
        print(f"Reference:  {row['reference']}")
        print(f"Prediction: {row['prediction']}")

    # Show best predictions
    print("\n📈 BEST PREDICTIONS (Highest Similarity):")
    best = results_df.nlargest(3, 'similarity')
    for idx, row in best.iterrows():
        print(f"\nSimilarity: {row['similarity']:.3f}")
        print(f"Transcription: {row['prediction']}")

    return results_df, final_score

In [ ]:
model_name = "./whisper-small-dialects"
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float32
).to("cuda")

In [ ]:
TEST_CSV_PATH = "/kaggle/input/shobdotori/Test_annotation/sample_submission.csv"
TEST_AUDIO_DIR = "/kaggle/input/shobdotori/Test"

# Run evaluation
results_df, final_score = evaluate_test_set(
    model=model,
    processor=processor,
    test_csv_path=TEST_CSV_PATH,
    test_audio_dir=TEST_AUDIO_DIR
)

# Save detailed results
results_df.to_csv("test_results_with_scores.csv", index=False, encoding='utf-8')
print("\n💾 Results saved to 'test_results_with_scores.csv'")